# lean — batched eval on Kaggle GPU

Same accuracy + mean-tokens measurement as `eval.py`, but batched on GPU
instead of sequential on CPU — should take a few minutes total instead of
~20-40 min per model.

1. New Notebook → Settings → Accelerator: **GPU T4 x2** (or P100), Internet on.
2. Add Data → upload the `eval_bundle/` folder as one Kaggle dataset (e.g. `lean-eval-bundle`) — it contains `eval.jsonl` plus each adapter's `v1/`, `v2/`, `v3/`, `v4/`.
3. Run all cells.

In [ ]:
!pip install -q transformers peft accelerate

In [ ]:
CFG = {
    "base_model": "Qwen/Qwen2.5-1.5B-Instruct",
    "bundle_dir": "/kaggle/input/datasets/johnandreimartinez/lean-eval-bundle",  # adjust to your uploaded dataset
    "adapters": ["v1", "v2", "v3", "v4"],
    "batch_size": 25,
    "max_new_tokens": 450,
}

In [ ]:
import json
import os
import re

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer


def extract_final_number(text):
    nums = re.findall(r"-?\d[\d,]*\.\d+|-?\d[\d,]*", text.replace(",", ""))
    return nums[-1] if nums else None


rows = [json.loads(l) for l in open(os.path.join(CFG["bundle_dir"], "eval.jsonl"), encoding="utf-8")]
gold = [r["answer"].split("####")[-1].strip().replace(",", "") for r in rows]
questions = [r["question"] for r in rows]
print("eval set:", len(rows), "questions")

In [ ]:
def batched_generate(model, tok, questions, batch_size, max_new_tokens):
    outputs = []
    with torch.no_grad():
        for start in range(0, len(questions), batch_size):
            batch = questions[start:start + batch_size]
            inputs = tok(batch, return_tensors="pt", padding=True).to("cuda")
            out = model.generate(
                **inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id,
            )
            gen_only = out[:, inputs["input_ids"].shape[1]:]
            for ids in gen_only:
                trimmed = ids[ids != tok.pad_token_id]
                outputs.append((tok.decode(trimmed, skip_special_tokens=True), len(trimmed)))
    return outputs


def run_eval(name, adapter_dir=None):
    tok = AutoTokenizer.from_pretrained(adapter_dir or CFG["base_model"])
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    base = AutoModelForCausalLM.from_pretrained(CFG["base_model"], torch_dtype=torch.float16, device_map="cuda")
    model = PeftModel.from_pretrained(base, adapter_dir) if adapter_dir else base
    model.eval()

    results = batched_generate(model, tok, questions, CFG["batch_size"], CFG["max_new_tokens"])
    correct = sum(1 for (text, _), g in zip(results, gold) if extract_final_number(text) == g)
    mean_tokens = sum(n for _, n in results) / len(results)
    print(f"{name}: accuracy={correct/len(results):.1%}  mean_output_tokens={mean_tokens:.1f}  n={len(results)}")

    del model, base
    torch.cuda.empty_cache()

In [ ]:
run_eval("base")
for name in CFG["adapters"]:
    run_eval(name, os.path.join(CFG["bundle_dir"], name))